In [21]:
import pandas as pd
import os
from tqdm import tqdm

tagged_df  = pd.read_excel('batchs/S_p_m.xlsx')
BATCH_SIZE = 50
os.makedirs('batchs/t_minus_batches', exist_ok=True)

batch, batch_idx = [], 0

def table_to_str(tbl: pd.DataFrame) -> str:
    """Компактная сериализация: строки через | , ячейки через ;"""
    header = ';'.join(tbl.columns)
    rows   = ['|'.join(str(v) for v in r) for r in tbl.values]
    return header + '\n' + '|'.join(rows)

for _, row in tqdm(tagged_df.iterrows(), total=len(tagged_df)):
    try:
        tbl = pd.read_csv(os.path.join('data', row['context']),
                          quotechar='"', on_bad_lines='skip', engine='python')
        batch.append({
            "id":       row['id'],
            "question": row['utterance'],
            "answer":   str(row['targetValue']),
            "table":    table_to_str(tbl)
        })

        if len(batch) == BATCH_SIZE:
            pd.DataFrame(batch).to_csv(
                f'batchs/t_minus_batches/batch_{batch_idx:04d}.tsv',
                sep='\t', index=False
            )
            batch_idx += 1
            batch = []

    except Exception as e:
        print(f"Пропуск id={row['id']}: {e}")

if batch:  # последний неполный батч
    pd.DataFrame(batch).to_csv(
        f'batchs/t_minus_batches/batch_{batch_idx:04d}.tsv',
        sep='\t', index=False
    )
    batch_idx += 1

print(f"✅ Батчей: {batch_idx} по {BATCH_SIZE} строк → batchs/t_minus_batches/")

100%|███████████████████████████████████████████████████████████████████████████| 14152/14152 [00:17<00:00, 795.54it/s]

✅ Батчей: 284 по 50 строк → batchs/t_minus_batches/


In [23]:
import pandas as pd
import os
import json
from tqdm import tqdm

tagged_df = pd.read_excel('batchs/S_p_m.xlsx')
BATCH_SIZE = 50
os.makedirs('batchs/t_minus_batches', exist_ok=True)

batch, batch_idx = [], 0

def table_to_str(tbl: pd.DataFrame) -> str:
    """Компактная сериализация: строки через | , ячейки через ;"""
    header = ';'.join(tbl.columns)
    rows = ['|'.join(str(v) for v in r) for r in tbl.values]
    return header + '\n' + '|'.join(rows)

for _, row in tqdm(tagged_df.iterrows(), total=len(tagged_df)):
    try:
        tbl = pd.read_csv(
            os.path.join('data', row['context']),
            quotechar='"', on_bad_lines='skip', engine='python'
        )
        batch.append({
            "id": row['id'],
            "question": row['utterance'],
            "answer": str(row['targetValue']),
            "table": table_to_str(tbl)
        })

        if len(batch) == BATCH_SIZE:
            # Сохраняем батч в JSON-файл
            with open(f'batchs/t_minus_batches/batch_{batch_idx:04d}.json', 'w', encoding='utf-8') as f:
                json.dump(batch, f, ensure_ascii=False, indent=2)
            batch_idx += 1
            batch = []

    except Exception as e:
        print(f"Пропуск id={row['id']}: {e}")

# Сохраняем последний неполный батч
if batch:
    with open(f'batchs/t_minus_batches/batch_{batch_idx:04d}.json', 'w', encoding='utf-8') as f:
        json.dump(batch, f, ensure_ascii=False, indent=2)
    batch_idx += 1

print(f"✅ Батчей: {batch_idx} по {BATCH_SIZE} строк → batchs/t_minus_batches/")

100%|███████████████████████████████████████████████████████████████████████████| 14152/14152 [00:22<00:00, 630.87it/s]


✅ Батчей: 284 по 50 строк → batchs/t_minus_batches/
